In [22]:
import pandas as pd
import re
import string
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report
import joblib
from newspaper.article import Article


In [23]:
df_fake = pd.read_csv('fake.csv')
df_authentic = pd.read_csv('authentic.csv')

In [24]:
df_fake.head()


,title,text,label
0,Aliens Found in Desert,Scientists have discovered alien life forms li...,FAKE
1,Man Builds Time Machine,A man claims he has invented a working time ma...,FAKE
2,Chocolate Cures Cancer,Eating chocolate daily can cure all types of c...,FAKE
3,Dinosaurs Spotted in Forest,Hikers report seeing live dinosaurs in a remot...,FAKE
4,City Will Be Underwater Tomorrow,A viral post claims that the city will be unde...,FAKE


In [25]:
df_authentic.head()


,title,text,label
0,Government Approves New Roads Project,The government has approved a new project to b...,REAL
1,World Leaders Meet to Discuss Economy,Leaders from major countries met to talk about...,REAL
2,Local Elections Dates Announced,The election commission announced the dates fo...,REAL
3,New Solar Energy Policy Released,The government introduced a policy to encourag...,REAL
4,Prime Minister Visits Flood Areas,The Prime Minister visited areas affected by f...,REAL


In [26]:
#Add labels (0 = fake, 1 = real)
df_fake["label"] = 0
df_authentic["label"] = 1



In [27]:
df = pd.concat([df_fake,df_authentic],ignore_index=True)

In [28]:
data = pd.concat([df_fake,df_authentic], axis = 0)


In [29]:
data = data.reset_index(drop=True)

In [30]:
data.sample(20)

,title,text,label
53,Parliament Debates Tax Changes,Lawmakers are discussing new tax rules to help...,1
57,Healthcare Funding Increased,Parliament approved more funding for hospitals...,1
26,New Virus Gives Superpowers,A viral post claims a new virus gives humans s...,0
33,Talking Dogs Take Over Jobs,Fake news claims dogs are trained to work as o...,0
44,Robots Take Over Schools,Fake news says robots will replace teachers in...,0
90,Streaming Platform Launches Originals,A new streaming service started releasing orig...,1
13,New Pill Makes You Fly,Fake news claims a pill can give humans the ab...,0
56,Digital ID System Launched,A new digital ID system was launched to make c...,1
14,President Resigns to Become Superhero,A fake post says the president resigned to fig...,0
73,Software Update Improves Security,A software update improves security and user e...,1


In [31]:
data = data.drop(["title","text"], axis = 1,errors='ignore')

In [32]:
data.reset_index(inplace=True)

In [33]:
data.drop(['index'],axis = 1,inplace=True)

In [34]:
data.sample(12)

,label
71,1
11,0
72,1
39,0
61,1
90,1
8,0
15,0
16,0
80,1


In [35]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"\[.*?\]", "", text)                      # remove things inside [ ]
    text = re.sub(r"http\S+|www\S+", "", text)               # remove links
    text = re.sub(r"<.*?>+", "", text)                       # remove HTML tags
    text = re.sub(r"[%s]" % re.escape(string.punctuation), "", text)  # remove punctuation
    text = re.sub(r"\n", "", text)                           # remove newlines
    text = re.sub(r"\w*\d\w*", "", text)                     # remove words with numbers
    text = re.sub(r"\s+", " ", text)                         # replace multiple spaces with single space
    return text


In [36]:
x = df["text"].apply(clean_text)
y = df["label"]


In [37]:
# Assuming 'label' is the column containing the news article text
x = df["text"]

# Assuming 'label' is the column containing the classification (e.g., 'fake' or 'real')
y = df["label"]

x_train, x_test, y_train, y_test = train_test_split(df["text"],df["label"],test_size=0.2,random_state=42) # type: ignore

In [38]:
vectorizer = TfidfVectorizer()
xv_train = vectorizer.fit_transform(x_train)
xv_test = vectorizer.transform(x_test)



In [39]:
nb = MultinomialNB()
nb.fit(xv_train,y_train)


,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [40]:
prediction = nb.predict(xv_test) 

In [41]:
print(classification_report(y_test,prediction,zero_division=0))

              precision    recall  f1-score   support

           0       1.00      0.92      0.96        13
           1       0.86      1.00      0.92         6

    accuracy                           0.95        19
   macro avg       0.93      0.96      0.94        19
weighted avg       0.95      0.95      0.95        19



In [42]:
joblib.dump(vectorizer,"vectorizer.pkl")
joblib.dump(nb,"lr_model.pkl")

['lr_model.pkl']